In [ ]:
import json
import torch
import random
import copy
import gc
import re
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm
from huggingface_hub import notebook_login
notebook_login()

# =====================================================================
#  REFERENCE-FUNCTION ABLATION inference
#  ---------------------------------------------------------------------
#  Identical to the baseline run EXCEPT the "Reference function: F" line
#  is removed from BOTH the few-shot examples AND the test query (the
#  "full-strip" design). The natural-language Task line is kept. The
#  ground-truth F for each test item is preserved in a "ref_function"
#  field in the output so the scorer can still judge hit/call/syntax.
#
#  A/B comparability: same seeds, same N-way K-shot sampling, same models.
#  The ONLY difference vs baseline is ABLATION=True -> F stripped.
# =====================================================================

# ================= Configuration =================
MODEL_ID_SET = [
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "google/gemma-2-2b-it",
    "google/gemma-3-1b-it",
    "google/gemma-4-e2b-it",
    "Qwen/Qwen2.5-Coder-0.5B-Instruct",
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen3.5-2B-Base"
]

TRAIN_SET_FILE = "../data/original/train_set_unified.jsonl"  # survey-era demonstration pool (1,376)
TEST_SET_FILE = "../data/original/test_set_150.jsonl"  # survey-era test set (150)

# 🌟 ABLATION SWITCH: True = strip "Reference function" everywhere (Group B)
#                     False = original baseline behaviour (Group A)
ABLATION = True

# ================= 0. Reference-function stripping =================
def strip_reference_function(content):
    """
    Remove the 'Reference function: X' line from a user-message content string.
    Returns (stripped_content, F). Handles \\r\\n endings and the $: tag.
    Keeps Dimension and Task lines intact.
    """
    F = None
    kept = []
    for line in content.split('\n'):
        m = re.match(r'\s*Reference function:\s*(.*?)\s*$', line)
        if m:
            F = m.group(1).strip()
            continue  # DROP this line
        kept.append(line)
    while kept and kept[-1].strip() == '':
        kept.pop()
    return '\n'.join(kept), F

# ================= 1. Data loading =================
def load_and_categorize_data(filepath):
    """
    Read the JSONL demonstration pool and group the items by Dimension.
    """
    categorized_data = defaultdict(list)

    print(f"📦 Loading and categorizing training data from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)

            user_content = ""
            for msg in data.get('messages', []):
                if msg.get('role') == 'user':
                    user_content = msg.get('content', '')
                    break

            # Use the Dimension field as the grouping key
            dimension = "Unknown"
            for text_line in user_content.split('\n'):
                if text_line.startswith('Dimension:'):
                    dimension = text_line.replace('Dimension:', '').strip()
                    break

            # Keep only the user/assistant pair (drop the system turn so it is not repeated)
            few_shot_pair = [msg for msg in data.get('messages', []) if msg['role'] in ['user', 'assistant']]

            if len(few_shot_pair) == 2:
                categorized_data[dimension].append(few_shot_pair)

    print("📊 Categories Distribution:")
    for dim, items in categorized_data.items():
        print(f"  - {dim}: {len(items)} items")

    return categorized_data

# ================= 2. N-way K-shot sampling =================
def generate_n_way_k_shot_prompt(categorized_data, K):
    """
    Strict N-way K-shot sampling (e.g. 8 dimensions x K=5 = 40 demonstrations).
    ABLATION: after sampling, remove the Reference function line from every demonstration's user turn.
    """
    sampled_pairs = []

    for dimension, examples in categorized_data.items():
        actual_k = min(K, len(examples))
        sampled = random.sample(examples, actual_k)
        sampled_pairs.extend(sampled)

    # Shuffle (user, assistant) pairs as units so questions and answers stay aligned
    random.shuffle(sampled_pairs)

    # Flatten into a single list of messages
    flat_few_shot_context = []
    for pair in sampled_pairs:
        # pair = [user_msg, assistant_msg]
        if ABLATION:
            pair = copy.deepcopy(pair)
            for msg in pair:
                if msg['role'] == 'user':
                    stripped, _ = strip_reference_function(msg['content'])
                    msg['content'] = stripped
        flat_few_shot_context.extend(pair)

    return flat_few_shot_context

# ================= 3. Inference =================
def generated(MODEL_ID, shared_few_shot_context, K_SHOT=1, seed=42, do_sample=False, temperature=0.7, top_p=0.9):

    # 🌟 ABLATION: output filename prefix switches to keep A/B files separate
    prefix = "ablation" if ABLATION else "baseline"
    OUTPUT_FILE = f"{prefix}_{MODEL_ID.split('/')[-1].replace('.', '_').lower()}_{K_SHOT}shot_seed{seed}.jsonl"

    SYSTEM_PROMPT = "You are a Strudel coding assistant. Write a Strudel code snippet for the following task. Return ONLY valid Strudel code, no markdown, no explanations."

    print(f"\nLoading model: {MODEL_ID}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        for line in tqdm(lines, desc=f"Evaluating {K_SHOT}-Shot (Seed {seed})"):
            if not line.strip(): continue
            data = json.loads(line)
            user_task = data['messages'][1]['content']

            # 🌟 ABLATION: strip the query's Reference function, keep the F for scoring
            ref_function = None
            if ABLATION:
                user_task, ref_function = strip_reference_function(user_task)
            else:
                # baseline: still record F so both files carry it identically
                _, ref_function = strip_reference_function(user_task)

            messages_context = copy.deepcopy(shared_few_shot_context)

            if "gemma-2" in MODEL_ID.lower():
                if len(messages_context) > 0:
                    messages_context[0]["content"] = f"{SYSTEM_PROMPT}\n\n{messages_context[0]['content']}"
                    messages_context.append({"role": "user", "content": f"Task: {user_task}"})
                    messages = messages_context
                else:
                    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nTask: {user_task}"}]
            else:
                messages = [{"role": "system", "content": SYSTEM_PROMPT}]
                messages.extend(messages_context)
                messages.append({"role": "user", "content": user_task})

            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

            # Reset the seed before each item so the whole batch is reproducible
            torch.manual_seed(seed)

            gen_kwargs = dict(
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=do_sample,
            )
            if do_sample:
                gen_kwargs.update(temperature=temperature, top_p=top_p)

            with torch.inference_mode():
                generated_ids = model.generate(**model_inputs, **gen_kwargs)

            generated_ids = [
                output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            clean_code = response.replace("```javascript", "").replace("```js", "").replace("```strudel", "").replace("```", "").strip()

            result_data = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_task},          # ablation: F already stripped
                    {"role": "assistant", "content": clean_code}
                ],
                "ref_function": ref_function                          # 🌟 ground-truth F for scoring
            }
            out_f.write(json.dumps(result_data, ensure_ascii=False) + "\n")

    print(f"✅ Results saved to: {OUTPUT_FILE}")
    del model; del tokenizer; gc.collect(); torch.cuda.empty_cache()

# ================= 4. Run loop (same demonstrations for every model) =================

# 1. Load the demonstration pool once
global_categorized_data = load_and_categorize_data(TRAIN_SET_FILE)

# 2. Experiment settings
CURRENT_K_SHOT = 0
SEED_LIST = [2, 4, 6, 8, 10]
USE_SAMPLING = (CURRENT_K_SHOT == 0)

# 3. Loop over shot counts and generation seeds
for current_seed in SEED_LIST:
    print(f"\n" + "="*60)
    print(f"🎯 STARTING NEW ITERATION: K-Shot={CURRENT_K_SHOT}, SEED={current_seed}, ABLATION={ABLATION}")
    print("="*60)

    # Build the demonstration context once per seed, before any model is loaded
    random.seed(current_seed)
    shared_few_shot_context = generate_n_way_k_shot_prompt(global_categorized_data, K=CURRENT_K_SHOT)
    print(shared_few_shot_context)
    print(f"✅ Generated ONE shared context with {len(shared_few_shot_context)//2} examples.")
    print(f"🔒 This exact context will be used for ALL {len(MODEL_ID_SET)} models in this seed iteration.")

    for MODEL_ID in MODEL_ID_SET:
        generated(MODEL_ID, shared_few_shot_context, K_SHOT=CURRENT_K_SHOT, seed=current_seed, do_sample=USE_SAMPLING, temperature=0.7, top_p=0.9)

print("\nAll runs finished.")